# Dish-use Analysis

This notebook analyzes the dish use of every fish in a 35-fish experiment. It divides the dishes in a grid and samples the coordinates of the fish every X frame of the video recording over a pre-difined time-window. Then it counts how many times the fish was sampled in a given bin of the grid. The analysis then uses these counts to calculate an "evenness" score of the ocuppancy of the fish across the dish and to generate a heatmap of dish use for every fish. 

## Settings

In [ ]:
from pathlib import Path
import glob
import hashlib
import json
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

repo_candidate = Path.cwd()
if repo_candidate.name == 'Analyses':
    repo_candidate = repo_candidate.parent
CODE_DIR = Path(os.environ.get('JLSOCIALBEHAVIOR_CODE_DIR', repo_candidate)).resolve()
os.chdir(CODE_DIR)

from functions.dish_use import analyze_dish_use

# Experiment index and animal metadata from the slim selection notebook. Edit these paths first for other datasets.
PROCESSING_SETTINGS_CSV = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use//temp/shoaling_selection/ShoalQTL_temp_processing/processingSettings.csv'
META_FOLDER = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/07_Data/Carlos/Behavior/QTL/'
META_FILE = 'MetaData_QTL.xlsx'
OUTPUT_DIR = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use//temp/shoaling_selection/ShoalQTL_temp_output/trajectory_grids/'

# Experiment selection. Folder wins when set; otherwise SELECT_EXPERIMENT_INDEX is used.
SELECT_EXPERIMENT_FOLDER = None #just the folder name, you don't need the full path. Set to None to use SELECT_EXPERIMENT_INDEX instead.
SELECT_EXPERIMENT_INDEX = 10

# Time window controls. Default is the second hour of experiment, when the shoaling stimuli are presented; set USE_FULL_RECORDING=True to ignore TIME_WINDOW_MIN.
TIME_WINDOW_MIN = (60, 120) # (start, end) in minutes. The first 60 minutes should be ignored since they are the acclimation period.
USE_FULL_RECORDING = False
FRAME_STRIDE = 30 #  We sample every Nth frame, were N is the value of FRAME_STRIDE. A value of 30 in a 30 fps videomeans we sample 1 frame per second. Set this variable to 1 to use all frames, but this will be computationally intensive and might not add much extra information.

# Data and output controls.
N_FISH = 35
FPS = 30
USE_TRAJECTORY_CACHE = False # Probably not needed anymore, but can be used to skip the trajectory calculations and use cached results if they exist. Might remove caching in the future.
SAVE_OUTPUTS = True
FIGURE_DPI = 180

# Dish grid and occupancy heatmap controls.
DISH_GRID_SIDE_LENGTH = 40 # Number of bins(grid cells) along each side of the dish grid. The grid is a square, so the total number of bins is (DISH_GRID_SIDE_LENGTH)^2. Afterwards bins whose center lies outside the circular dish boundary are filtered out by dish_use.py.
OCCUPANCY_HEATMAP_CMAP = 'viridis' # Use one shared color scale across all 35 fish so bright panels are comparable.
OCCUPANCY_HEATMAP_VMAX_PERCENTILE = 99
DISH_EVENNESS_ZMAD_THRESHOLD = 1.5

os.makedirs(OUTPUT_DIR, exist_ok=True)
CACHE_DIR = os.path.join(OUTPUT_DIR, 'cache')
os.makedirs(CACHE_DIR, exist_ok=True)

print('Using repository:', CODE_DIR)
print('Using processing settings:', PROCESSING_SETTINGS_CSV)
print('Using animal metadata:', os.path.join(META_FOLDER, META_FILE))
print('Writing trajectory outputs to:', OUTPUT_DIR)

Using repository: C:\Users\crodri19\larschpostdoc_analyses\selection_shoaling\jlsocialbehavior
Using processing settings: //nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use//temp/shoaling_selection/ShoalQTL_temp_processing/processingSettings.csv
Using animal metadata: //nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/07_Data/Carlos/Behavior/QTL/MetaData_QTL.xlsx
Writing trajectory outputs to: //nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use//temp/shoaling_selection/ShoalQTL_temp_output/trajectory_grids/


## Helpers

In [ ]:
# This function makes labels safe to use in filenames by replacing spaces,
# slashes, and other special characters with underscores.
def safe_filename(value):
    text = str(value).strip()
    return ''.join(char if char.isalnum() or char in '._-' else '_' for char in text)


## Select Experiment

In [ ]:
# Load the processing table that lists all experiments.
experiments = pd.read_csv(PROCESSING_SETTINGS_CSV, index_col=0)
if experiments.empty:
    raise ValueError('No experiments found in ' + PROCESSING_SETTINGS_CSV)

# Load animal metadata once. The trajectory-loading cell will use this table
# after it knows which animal IDs are present in the selected experiment.
meta_path = os.path.join(META_FOLDER, META_FILE)
if os.path.exists(meta_path):
    infoAn = pd.read_excel(meta_path, sheet_name='AllAn')
else:
    warnings.warn('Animal metadata file not found; subplot genotypes will be marked unknown: ' + meta_path)
    infoAn = pd.DataFrame(columns=['anNr', 'genotype'])

# Choose the experiment row. You can either use the explicit folder name or the numeric index displayed on the table.
# The explicit folder name is safer for repeated use;
# The numeric index is convenient for quick browsing.
if SELECT_EXPERIMENT_FOLDER is not None:
    matches = experiments[experiments['folder'].astype(str) == str(SELECT_EXPERIMENT_FOLDER)]
    if matches.empty:
        raise ValueError(f'SELECT_EXPERIMENT_FOLDER not found: {SELECT_EXPERIMENT_FOLDER}')
    selected_row = matches.iloc[0]
    selected_index = matches.index[0]
else:
    selected_index = experiments.index[SELECT_EXPERIMENT_INDEX]
    selected_row = experiments.loc[selected_index]

# Show a compact preview so the selected row can be checked before loading data.
preview_cols = [c for c in ['folder', 'date', 'setup', 'txtPath'] if c in experiments.columns]
display(experiments[preview_cols].reset_index().rename(columns={'index': 'processing_row'}).head(30))

print('Selected processing row:', selected_index)
print('Selected folder:', selected_row.get('folder', ''))
print('Raw trajectory file:', selected_row['txtPath'])


,processing_row,folder,date,setup,txtPath
0,0,20260511_q1_Sel1A_s1,11-05-2026,1,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...
1,1,20260511_q2_Sel1A_s2,11-05-2026,2,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...
2,2,20260512_q3_Sel3C_s1,12-05-2026,1,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...
3,3,20260512_q4_Sel3C_s2,12-05-2026,2,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...
4,4,20260518_q5_Sel3D_s1,18-05-2026,1,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...
5,5,20260518_q6_Sel3D_s2,18-05-2026,2,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...
6,6,20260519_q7_Sel1B_s1,19-05-2026,1,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...
7,7,20260519_q8_Sel1B_s2,19-05-2026,2,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...
8,8,20260626_q9_Sel1A_s2,26-06-2026,2,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...
9,9,20260731_q10_Sel3A_s1,31-07-2026,1,\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsc...


Selected processing row: 10
Selected folder: 20260731_q11_Sel3A_s2
Raw trajectory file: \\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsch\default\D2c\07_Data\Carlos\Behavior\QTL\20260731_q11_Sel3A_s2\PositionTxt_allROI2026-07-31T10_52_17.csv


## Load Trajectory Window
Trajectory caches may contain `dish_radius_mm` or the legacy `arena_radius_mm` field. The reader prefers `dish_radius_mm`; new caches write only that field. Raw trajectories require an ROI file. A readable cache can be used without an ROI, with a warning.


In [ ]:
# Get the trajectory data only for the specified time window of the experiment.
# Convert the requested time window from minutes to raw video frames.
# When USE_FULL_RECORDING=True, n_rows stays None so pandas/numpy read to the end.
if USE_FULL_RECORDING or TIME_WINDOW_MIN is None:
    start_frame = 0
    n_rows = None
    window_label = 'full_recording'
else:
    start_min, stop_min = TIME_WINDOW_MIN
    if start_min < 0 or stop_min <= start_min:
        raise ValueError('TIME_WINDOW_MIN must be a (start, stop) tuple in minutes.')
    start_frame = int(round(start_min * 60 * FPS))
    stop_frame = int(round(stop_min * 60 * FPS))
    n_rows = stop_frame - start_frame
    window_label = f'{start_min:g}-{stop_min:g}_min'

# Build the same cache key as before: selected raw file, time window, stride,
# fish count, and raw-file modification time. Changing any of these creates a
# new cache file instead of silently reusing stale coordinates.
txt_path = str(selected_row['txtPath'])
experiment_label = selected_row.get('folder', Path(txt_path).stem)
try:
    raw_mtime = os.path.getmtime(txt_path)
except OSError:
    raw_mtime = None
cache_key = {
    'txt_path': txt_path,
    'txt_stem': Path(txt_path).stem,
    'experiment_label': str(experiment_label),
    'start_frame': start_frame,
    'n_rows': n_rows,
    'frame_stride': int(FRAME_STRIDE),
    'n_fish': int(N_FISH),
    'mtime': raw_mtime,
}
digest = hashlib.sha1(json.dumps(cache_key, sort_keys=True, default=str).encode('utf-8')).hexdigest()[:12]
cache_label = safe_filename(f'{experiment_label}_{Path(txt_path).stem}_{digest}')
trajectory_cache = os.path.join(CACHE_DIR, cache_label + '.npz')

# Look for the ROI even on cache hits so missing geometry files are reported.
start_dir = os.path.dirname(txt_path)
roi_matches = sorted(
    path for path in glob.glob(os.path.join(start_dir, 'ROIdef*'))
    if os.path.isfile(path)
)
roi_path = roi_matches[0] if roi_matches else None

if USE_TRAJECTORY_CACHE and os.path.exists(trajectory_cache):
    # Cached files already contain millimeter-centered coordinates and metadata.
    cached = np.load(trajectory_cache, allow_pickle=False)
    xy_mm = cached['xy_mm']
    animal_ids = cached['animal_ids']
    # Read the current geometry field or its legacy arena-use name.
    if 'dish_radius_mm' in cached:
        radius_key = 'dish_radius_mm'
    elif 'arena_radius_mm' in cached:
        radius_key = 'arena_radius_mm'
    else:
        raise ValueError(
            f'Trajectory cache {trajectory_cache} is missing both '
            'dish_radius_mm and legacy arena_radius_mm geometry fields.'
        )
    dish_radius_mm = float(cached[radius_key])
    trajectory_meta = json.loads(str(cached['meta_json']))
    print('Loaded cached trajectory window:', trajectory_cache)
    if roi_path is None:
        warnings.warn(
            f'No ROIdef* file found in {start_dir or "."}. '
            f'Using cached centered coordinates and dish geometry from {trajectory_cache}.'
        )
else:
    if roi_path is None:
        raise FileNotFoundError(
            f'No ROIdef* file found in {start_dir or "."} for trajectory {txt_path}. '
            'An ROI file is required when loading raw trajectories.'
        )

    # ROI radii define pixels/mm and the center of each dish's cropped image.
    dish_diameter_mm = float(selected_row.get('dishDiameter_mm', 70))
    try:
        rois = np.atleast_2d(np.loadtxt(roi_path))
    except Exception:
        rois = np.atleast_2d(np.loadtxt(roi_path, delimiter=','))
    radii_px = rois[:N_FISH, -1].astype(float)
    px_per_mm = 2 * np.nanmean(radii_px) / dish_diameter_mm
    centers_px = radii_px + 2  # The video crop extends 2 px beyond the circle ROI.
    dish_radius_mm = dish_diameter_mm / 2
    scale_source = roi_path

    # Parse the local animal IDs from the processing row. If no explicit IDs are
    # present, use 0..N_FISH-1 so every subplot still has a stable label.
    animal_id_source = None
    if 'anIDAll' in selected_row and pd.notna(selected_row['anIDAll']):
        animal_id_source = selected_row['anIDAll']
    elif 'anNr' in selected_row and pd.notna(selected_row['anNr']):
        animal_id_source = selected_row['anNr']

    if animal_id_source is None:
        animal_ids = np.arange(N_FISH)
    else:
        animal_id_text = str(animal_id_source).strip()
        if not animal_id_text or animal_id_text.lower() == 'nan':
            animal_ids = np.arange(N_FISH)
        elif ':' in animal_id_text:
            start_id, stop_id = animal_id_text.split(':', 1)
            animal_ids = np.arange(int(start_id), int(stop_id) + 1)
        else:
            animal_ids = np.array(re.split(r'[\s,]+', animal_id_text), dtype=int)
        if len(animal_ids) < N_FISH:
            raise ValueError(f'Animal ID list has {len(animal_ids)} entries but N_FISH={N_FISH}.')
        animal_ids = animal_ids[:N_FISH]

    # Read only x/y columns for the selected fish. Each fish contributes three raw
    # columns; this notebook uses the first two as x and y and skips the third.
    xy_cols = [col for fish in range(N_FISH) for col in (fish * 3, fish * 3 + 1)]
    if txt_path[-3:].lower() == 'npy':
        raw = np.load(txt_path)
        stop = None if n_rows is None else start_frame + n_rows
        raw_xy = raw[start_frame:stop:FRAME_STRIDE, :N_FISH, :2].astype(float)
    else:
        first_line = pd.read_csv(txt_path, header=None, nrows=1, sep=':')
        first_value = str(first_line.values[0][0])
        first_char = first_value[0] if first_value else ''

        if first_char == '(':
            raise ValueError('Old Bonsai format is not supported by this fast 35-fish trajectory reader.')
        if first_char == 'X':
            skiprows = [0] + list(range(1, start_frame + 1)) if start_frame else [0]
        else:
            skiprows = range(start_frame) if start_frame else None

        raw_xy_table = pd.read_csv(
            txt_path,
            header=None,
            delim_whitespace=True,
            usecols=xy_cols,
            skiprows=skiprows,
            nrows=n_rows,
        ).to_numpy(dtype=float)
        raw_xy_table = raw_xy_table[::FRAME_STRIDE]
        raw_xy = raw_xy_table.reshape(raw_xy_table.shape[0], N_FISH, 2)

    # Center every fish's x/y coordinates on its dish and convert pixels to mm.
    xy_mm = (raw_xy - centers_px[np.newaxis, :, np.newaxis]) / px_per_mm
    trajectory_meta = {
        'folder': str(selected_row.get('folder', '')),
        'txt_path': txt_path,
        'txt_stem': Path(txt_path).stem,
        'window_label': window_label,
        'start_frame': start_frame,
        'n_rows_requested': n_rows,
        'frame_stride': int(FRAME_STRIDE),
        'fps': int(FPS),
        'px_per_mm': float(px_per_mm),
        'scale_source': str(scale_source),
        'cache_key': cache_key,
    }

    if USE_TRAJECTORY_CACHE:
        np.savez_compressed(
            trajectory_cache,
            xy_mm=xy_mm,
            animal_ids=np.asarray(animal_ids),
            dish_radius_mm=np.asarray(dish_radius_mm),
            meta_json=np.asarray(json.dumps(trajectory_meta, default=str, sort_keys=True)),
        )
        print('Cached trajectory window:', trajectory_cache)
    else:
        print('Trajectory cache disabled; parsed raw trajectory file.')

# Match plotted animal IDs to genotype labels from the metadata sheet.
if infoAn is None or infoAn.empty:
    animal_genotypes = np.array(['unknown'] * len(animal_ids), dtype=object)
else:
    missing_cols = {'anNr', 'genotype'} - set(infoAn.columns)
    if missing_cols:
        raise ValueError('Animal metadata is missing required columns: ' + ', '.join(sorted(missing_cols)))

    metadata = infoAn[['anNr', 'genotype']].copy()
    metadata['anNr'] = pd.to_numeric(metadata['anNr'], errors='coerce')
    metadata = metadata.dropna(subset=['anNr']).drop_duplicates('anNr', keep='last')
    genotype_by_animal = metadata.set_index('anNr')['genotype']

    genotype_labels = []
    for animal_id in animal_ids:
        animal_key = pd.to_numeric(pd.Series([animal_id]), errors='coerce').iloc[0]
        genotype = genotype_by_animal.get(animal_key, np.nan)
        if pd.isna(genotype) or str(genotype).strip() == '':
            genotype_labels.append('unknown')
        else:
            genotype_labels.append(str(genotype).strip())
    animal_genotypes = np.asarray(genotype_labels, dtype=object)

print('xy_mm shape:', xy_mm.shape)
print('animal IDs:', animal_ids)
print('animal genotypes:', animal_genotypes)
print('dish radius (mm):', dish_radius_mm)
print('scale source:', trajectory_meta.get('scale_source'))


## Analyze Dish-Use Occupancy

Call `analyze_dish_use`, imported from `functions.dish_use`, to get the dish occupancy and its evenness metric per fish. It receives `xy_mm` from the trajectory-loading cell. Its shape is `(samples, fish, 2)`, with x and y in the last axis. The helper keeps the same fish order and should work with any number of fish; this notebook displays 35 fish in a 5x7 grid.

The next code cell sets `grid_side_length = DISH_GRID_SIDE_LENGTH` and uses the following call:

```python
dish_use = analyze_dish_use(
    xy_mm,
    dish_radius_mm=dish_radius_mm,
    grid_side_length=grid_side_length,
)
```

`dish_radius_mm` must be positive and finite. `grid_side_length` is an integer of at least two and specifies the number of cells along each axis, rather than a physical length in millimeters: a value of 40 creates a 40x40 square grid before circular masking. Positions with nonfinite x/y or outside the dish are excluded. A bin is eligible when its center lies inside or on the dish boundary. A position inside the dish can still land in an excluded edge bin.

Each retained position contributes one count. With `FRAME_STRIDE > 1`, these counts describe sampled positions, not every video frame. The helper does not estimate elapsed seconds or weight samples by time.

The returned `dishUseResult` has the fields below. Let `F` be the number of fish and `N` be `grid_side_length`. Map rows index y and columns index x, so `occupancy_frame_counts[fish, y_bin, x_bin]` accesses one bin. Grid geometry is shared by all fish.

| Field | Shape / type | Meaning and units |
| --- | --- | --- |
| `bin_edges` | `(N + 1,)` | Shared x/y boundaries from `-dish_radius_mm` to `+dish_radius_mm`, in mm. |
| `grid_x`, `grid_y` | Each `(N, N)` | x/y coordinates of bin centers, in mm. |
| `dish_mask` | `(N, N)`, boolean | eligible-bin mask based on bin centers. |
| `occupancy_frame_counts` | `(F, N, N)`, float | Counts of retained positions; float storage accommodates `NaN`. |
| `probability` | `(F, N, N)` | Counts divided by each fish's retained total; dimensionless. |
| `evenness` | `(F,)` | Normalized Shannon evenness; dimensionless. |
| `bias_index` | `(F,)` | `1 - evenness`; dimensionless. |
| `dish_bin_count` | Integer | Number of eligible bins, including unvisited bins; common to all fish. |
| `visited_bin_count` | `(F,)`, integer | eligible bins with at least one retained position. |
| `coverage_fraction` | `(F,)` | Visited bins divided by eligible bins; dimensionless. |
| `total_binned_frame_count` | `(F,)`, integer | Total retained positions in eligible bins. |
| `valid_in_dish_samples` | `(F,)`, integer | Finite positions inside/on the circle before bin masking; can exceed the retained count. |

Eligible unvisited bins contain zero occupancy; excluded bins contain `NaN`. For a fish with retained positions, eligible probabilities sum to one and unvisited bins have probability zero. With no retained positions, visited/total counts and coverage are zero, while the probability map, evenness, and bias index are `NaN`. If the grid has at most one eligible bin, probability and evenness also remain undefined.


In [ ]:
# Calculate occupancy and evenness across the dish  for every fish.
n_fish = xy_mm.shape[1]
n_rows, n_cols = 5, 7
if n_fish != n_rows * n_cols:
    raise ValueError(f'Expected {n_rows * n_cols} fish for a 5x7 grid, got {n_fish}.')
if len(animal_genotypes) != n_fish:
    raise ValueError(f'Expected {n_fish} genotype labels, got {len(animal_genotypes)}.')

grid_side_length = DISH_GRID_SIDE_LENGTH
cmap = OCCUPANCY_HEATMAP_CMAP
vmax_percentile = OCCUPANCY_HEATMAP_VMAX_PERCENTILE

dish_use = analyze_dish_use(
    xy_mm,
    dish_radius_mm=dish_radius_mm,
    grid_side_length=grid_side_length,
)
edges = dish_use.bin_edges
heatmaps = dish_use.occupancy_frame_counts
extent = [edges[0], edges[-1], edges[0], edges[-1]]


## Build Dish-Use Metrics and Screen Low Evenness

For each fish, let `p` be its bin probabilities and `K = dish_use.dish_bin_count` be the common number of eligible bins (bin centers inside or on the circular dish boundary). The helper calculates:

```text
evenness = -sum(p * log(p)) / log(K)
coverage = visited_bin_count / K
bias_index = 1 - evenness
```

The entropy sum uses positive probabilities and natural logarithms. Unvisited eligible bins have `p = 0` and remain in `K`. Evenness is zero when all retained positions occupy one bin and one when counts are equal across every eligible bin. Coverage describes how many bins were visited, while evenness also reflects how counts are distributed among them. The bias index is the complement of evenness and does not describe a direction of preference.

The metrics cell maps helper results to the existing table columns:

| Helper field | `dish_metrics` column(s) |
| --- | --- |
| `dish_bin_count` | `reachable_bin_count` |
| `visited_bin_count` | `visited_reachable_bin_count` |
| `total_binned_frame_count` | `total_binned_frame_count` |
| `evenness` | `dish_evenness_shannon` |
| `bias_index` | `dish_bias_index` |
| `coverage_fraction` | `dish_coverage_fraction`, `reachable_coverage_fraction` (identical values) |

The `reachable` names refer to the fixed geometric bin mask. `fish_idx` is the zero-based fish axis of `xy_mm`; `animal_id` and `genotype` come from metadata. `valid_samples` is calculated by the notebook and counts **all finite x/y positions**, including those outside the dish. `valid_in_dish_samples` counts positions inside the circle before bin masking; it is available on the helper result but is not exported in `dish_metrics`.

The notebook adds `dish_evenness_z_mad` and `low_dish_evenness_flag` after the helper call. Scores compare each fish with all fish in this experiment and time window, pooled across genotypes:

```text
zMAD = 0.6745 * (evenness - cohort_median) / MAD
MAD = median(abs(evenness - cohort_median))
```

Missing evenness values keep `NaN` scores and are not flagged. If MAD is zero or unavailable, nonmissing values receive a score of zero. The low-evenness flag is set when `zMAD <= -DISH_EVENNESS_ZMAD_THRESHOLD`; it is a cohort-relative screening criterion. The displayed table puts flagged fish first, then sorts by increasing zMAD, while the underlying table and exports retain fish order.


In [ ]:
# Build one per-fish table directly from the dish-use helper results.
required_dish_vars = ['dish_use', 'dish_radius_mm', 'xy_mm', 'animal_ids', 'animal_genotypes']
missing_dish_vars = [name for name in required_dish_vars if name not in globals()]
if missing_dish_vars:
    raise ValueError('Run the dish-use occupancy cell first; missing: ' + ', '.join(missing_dish_vars))

# Compare a metric with the median across fish in this experiment/window.
# MAD = median(abs(value - median)); zMAD = 0.6745 * (value - median) / MAD.
# The factor gives a standard-deviation-like scale for normally distributed data.
# Missing entries are omitted from the reference and retain NaN scores. If MAD
# is zero or unavailable, nonmissing entries receive 0 (no outlier flag).
def _robust_z_mad(values):
    values = pd.Series(values, dtype='float64')
    median = values.median(skipna=True)
    mad = (values - median).abs().median(skipna=True)
    if pd.isna(mad) or mad == 0:
        return pd.Series(np.where(values.notna(), 0.0, np.nan), index=values.index, dtype='float64')
    return 0.6745 * (values - median) / mad


n_fish = len(dish_use.evenness)
dish_metrics = pd.DataFrame({
    'fish_idx': np.arange(n_fish),
    'animal_id': animal_ids,
    'genotype': [str(value).strip() or 'unknown' for value in animal_genotypes],
    # Finite positions include positions outside the dish; binned counts do not.
    'valid_samples': np.isfinite(xy_mm).all(axis=2).sum(axis=0),
    'dish_coverage_fraction': dish_use.coverage_fraction,
    'reachable_bin_count': dish_use.dish_bin_count,
    'visited_reachable_bin_count': dish_use.visited_bin_count,
    'total_binned_frame_count': dish_use.total_binned_frame_count,
    'dish_evenness_shannon': dish_use.evenness,
    'dish_bias_index': dish_use.bias_index,
    'reachable_coverage_fraction': dish_use.coverage_fraction,
})

# Screen relative to ALL fish here, pooled across genotypes. This flag is a
# cohort-relative screening cutoff, not a p-value or fixed biological limit.
# Thresholds come from Settings. Comparisons with NaN return False, so an
# unscorable fish is not flagged by these comparisons; check its raw metric too.
dish_metrics['dish_evenness_z_mad'] = _robust_z_mad(dish_metrics['dish_evenness_shannon'])
dish_metrics['low_dish_evenness_flag'] = dish_metrics['dish_evenness_z_mad'] <= -DISH_EVENNESS_ZMAD_THRESHOLD


# Sort a display copy: low-evenness flags first, then lower evenness zMAD.
# This ranks fish for inspection; no fish are removed from dish_metrics.
dish_evenness_screen = dish_metrics.copy()
dish_evenness_screen = dish_evenness_screen.sort_values(
    ['low_dish_evenness_flag', 'dish_evenness_z_mad'],
    ascending=[False, True],
).reset_index(drop=True)

# SAVE_OUTPUTS also controls this CSV export. Save the full dish-use metrics table
# in its original order; the ranked display below uses dish_evenness_screen.
folder = trajectory_meta.get('folder') or trajectory_meta.get('txt_stem', '')
evenness_output_base = os.path.join(
    OUTPUT_DIR,
    safe_filename(f'occupancy_heatmap_grid_{folder}_{trajectory_meta.get("window_label", "window")}')
)
if SAVE_OUTPUTS:
    evenness_csv = evenness_output_base + '_dish_evenness.csv'
    dish_metrics.to_csv(evenness_csv, index=False)
    print('Wrote:', evenness_csv)

print('Expected dish radius (mm):', dish_radius_mm)
print('Expected-dish bin count:', dish_use.dish_bin_count)
print('Low dish-evenness flags:', int(dish_metrics['low_dish_evenness_flag'].sum()))

display_columns = [
    'fish_idx',
    'animal_id',
    'genotype',
    'dish_evenness_shannon',
    'dish_bias_index',
    'dish_evenness_z_mad',
    'low_dish_evenness_flag',
    'reachable_coverage_fraction',
    'reachable_bin_count',
    'visited_reachable_bin_count',
]
display(dish_evenness_screen[display_columns])


## Plot Dish-Use Heatmaps


In [ ]:
# Plot sampled-frame occupancy with per-fish evenness labels.
required_heatmap_label_columns = {
    'fish_idx',
    'animal_id',
    'genotype',
    'dish_evenness_shannon',
    'dish_evenness_z_mad',
}
missing_heatmap_label_columns = required_heatmap_label_columns - set(dish_metrics.columns)
if missing_heatmap_label_columns:
    raise ValueError('dish_metrics is missing columns: ' + ', '.join(sorted(missing_heatmap_label_columns)))

# Pick one color scale for all 35 heatmaps, using only positive in-dish bins.
# This makes bright panels comparable across fish without one extreme bin washing
# out the rest of the experiment.
positive_values = heatmaps[np.isfinite(heatmaps) & (heatmaps > 0)]
vmax = float(np.nanpercentile(positive_values, vmax_percentile)) if positive_values.size else 1.0
if not np.isfinite(vmax) or vmax <= 0:
    vmax = 1.0

heatmap_fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 10), sharex=True, sharey=True, dpi=FIGURE_DPI)
theta = np.linspace(0, 2 * np.pi, 181)
circle_x = dish_radius_mm * np.cos(theta)
circle_y = dish_radius_mm * np.sin(theta)
im = None

for fish_idx, ax in enumerate(axes.ravel()):
    im = ax.imshow(
        np.ma.masked_invalid(heatmaps[fish_idx]),
        origin='lower',
        extent=extent,
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        interpolation='nearest',
        rasterized=True,
    )
    ax.plot(circle_x, circle_y, color='white', lw=0.55, alpha=0.8)
    ax.axhline(0, color='white', lw=0.25, alpha=0.35, zorder=2)
    ax.axvline(0, color='white', lw=0.25, alpha=0.35, zorder=2)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_aspect('equal', adjustable='box')

    # Identify each fish and show its evenness and cohort-relative zMAD.
    row = dish_metrics.iloc[fish_idx]
    ax.set_title(
        f'{fish_idx:02d} | an {row["animal_id"]}\n'
        f'genotype: {row["genotype"]}\n'
        f'evenness {row["dish_evenness_shannon"]:.2f} | zMAD {row["dish_evenness_z_mad"]:.1f}',
        fontsize=6.0,
    )
    ax.tick_params(labelsize=6, length=2)

for ax in axes[-1, :]:
    ax.set_xlabel('x (mm)', fontsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel('y (mm)', fontsize=8)

folder = trajectory_meta.get('folder') or trajectory_meta.get('txt_stem', '')
heatmap_fig.suptitle(
    f'Sampled-frame occupancy: {folder} | {trajectory_meta.get("window_label", "window")} | '
    f'stride {trajectory_meta.get("frame_stride")} | shared vmax {vmax_percentile:g}th pct = {vmax:.1f} frames/bin',
    fontsize=12,
    y=0.995,
)
heatmap_fig.subplots_adjust(left=0.04, right=0.88, bottom=0.05, top=0.92, wspace=0.15, hspace=0.58)

# Put the shared colorbar in a fixed outer axis so it does not steal space from
# the rightmost subplots.
cbar_ax = heatmap_fig.add_axes([0.915, 0.12, 0.012, 0.74])
cbar = heatmap_fig.colorbar(im, cax=cbar_ax, label='sampled frames per bin')
cbar.ax.tick_params(labelsize=8)

output_base = os.path.join(
    OUTPUT_DIR,
    safe_filename(f'occupancy_heatmap_grid_{folder}_{trajectory_meta.get("window_label", "window")}')
)
if SAVE_OUTPUTS:
    heatmap_fig.savefig(output_base + '.png', dpi=FIGURE_DPI, bbox_inches='tight')
    heatmap_fig.savefig(output_base + '.pdf', bbox_inches='tight')
    dish_metrics.to_csv(output_base + '_metrics.csv', index=False)
    print('Wrote:', output_base + '.png')
    print('Wrote:', output_base + '.pdf')
    print('Wrote:', output_base + '_metrics.csv')

plt.show()


## Plot Dish-Use Evenness by Genotype

Each point represents one fish's normalized Shannon evenness. Dotted lines show the pooled cohort's positive and negative zMAD thresholds; low-evenness screening uses only the negative threshold.


In [ ]:
# Plot each fish's dish-use evenness from the dish_metrics table generated above.
required_evenness_plot_columns = {
    'fish_idx',
    'animal_id',
    'genotype',
    'dish_evenness_shannon',
    'dish_evenness_z_mad',
}
missing_evenness_plot_columns = required_evenness_plot_columns - set(dish_metrics.columns)
if missing_evenness_plot_columns:
    raise ValueError('dish_metrics is missing columns: ' + ', '.join(sorted(missing_evenness_plot_columns)))

individual_evenness = dish_metrics.copy()
individual_evenness['genotype'] = individual_evenness['genotype'].astype(str).str.strip().replace('', 'unknown')
individual_evenness = individual_evenness[np.isfinite(individual_evenness['dish_evenness_shannon'])].copy()
genotype_order = sorted(individual_evenness['genotype'].unique())
genotype_counts = individual_evenness.groupby('genotype').size().reindex(genotype_order)
palette_name = 'tab10' if len(genotype_order) <= 10 else 'tab20'
genotype_colors = dict(zip(genotype_order, sns.color_palette(palette_name, n_colors=len(genotype_order))))

np.random.seed(7)
fig_width = max(6, 1.2 * len(genotype_order) + 2)
fig, ax = plt.subplots(figsize=(fig_width, 4.5), dpi=FIGURE_DPI)

sns.stripplot(
    data=individual_evenness,
    x='genotype',
    y='dish_evenness_shannon',
    hue='genotype',
    order=genotype_order,
    hue_order=genotype_order,
    palette=genotype_colors,
    jitter=0.13,
    size=7.5,
    edgecolor='white',
    linewidth=0.6,
    alpha=0.9,
    legend=False,
    ax=ax,
    zorder=3,
)

evenness_values_for_plot = pd.Series(individual_evenness['dish_evenness_shannon'], dtype='float64')
evenness_median_for_plot = evenness_values_for_plot.median(skipna=True)
evenness_mad_for_plot = (evenness_values_for_plot - evenness_median_for_plot).abs().median(skipna=True)
if not pd.isna(evenness_mad_for_plot) and evenness_mad_for_plot != 0:
    zmad_delta = DISH_EVENNESS_ZMAD_THRESHOLD * evenness_mad_for_plot / 0.6745
    zmad_evenness_lines = [
        (f'zMAD +{DISH_EVENNESS_ZMAD_THRESHOLD:g}', evenness_median_for_plot + zmad_delta, '#8B1E3F'),
        (f'zMAD -{DISH_EVENNESS_ZMAD_THRESHOLD:g}', evenness_median_for_plot - zmad_delta, '#1F5A7A'),
    ]
    for zmad_label, evenness_threshold, line_color in zmad_evenness_lines:
        if np.isfinite(evenness_threshold):
            ax.axhline(
                evenness_threshold,
                color=line_color,
                linestyle=':',
                linewidth=1.8,
                label=f'{zmad_label} = {evenness_threshold:.2f}',
                zorder=2,
            )

ax.set_xticks(range(len(genotype_order)))
ax.set_xticklabels([f'{genotype}\n(n={genotype_counts[genotype]})' for genotype in genotype_order], rotation=25, ha='right')
ax.set_ylabel('dish-use evenness')
ax.set_xlabel('genotype')
ax.set_title(
    f'Individual dish-use evenness: {trajectory_meta.get("folder") or trajectory_meta.get("txt_stem", "")}'
    f' | {trajectory_meta.get("window_label", "window")}'
)
#ax.set_ylim(0.8, 0.9)
ax.grid(axis='y', color='0.88', linewidth=0.8)
sns.despine(ax=ax)
ax.legend(frameon=False, loc='best', fontsize=8)
fig.tight_layout()

if SAVE_OUTPUTS:
    folder = trajectory_meta.get('folder') or trajectory_meta.get('txt_stem', '')
    output_base = os.path.join(
        OUTPUT_DIR,
        safe_filename(f'individual_dish_evenness_distribution_{folder}_{trajectory_meta.get("window_label", "window")}')
    )
    fig.savefig(output_base + '.png', dpi=FIGURE_DPI, bbox_inches='tight')
    fig.savefig(output_base + '.pdf', dpi=FIGURE_DPI, bbox_inches='tight')
    individual_evenness.to_csv(output_base + '_dish_evenness_distribution.csv', index=False)
    print('Wrote:', output_base + '.png')
    print('Wrote:', output_base + '.pdf')
    print('Wrote:', output_base + '_dish_evenness_distribution.csv')

display(individual_evenness.sort_values('dish_evenness_shannon', ascending=True).reset_index(drop=True))
plt.show()
